In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import os

# Locate repo root — works whether running locally or on Binder/Colab
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path('../../'))
sys.path.insert(0, str(_root / 'src'))

from data_loader import ensure_data, PROCESSED_DIR
ensure_data()

# X_train.csv / y_train.csv / X_test.csv are generated by home_credit_pipeline.ipynb
# Run that notebook first if these files are missing.
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')

In [9]:
# lets see the dimensions of the data
print(f'Training data shape: {X_train.shape}')
print(f'Training labels shape: {y_train.shape}')
print(f'Testing data shape: {X_test.shape}')


Training data shape: (307511, 75)
Training labels shape: (307511, 1)
Testing data shape: (48744, 75)


In [16]:
base_rf = RandomForestClassifier(n_estimators=500, max_depth=15, random_state=42, n_jobs=-1)
base_rf.fit(X_train, y_train)

importances = pd.Series(base_rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

print(importances.head(20))


c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


EXT_SOURCE_3                    0.119436
EXT_SOURCE_2                    0.114237
EXT_SOURCE_1                    0.059826
DAYS_BIRTH                      0.046437
DAYS_ID_PUBLISH                 0.041393
DAYS_REGISTRATION               0.039447
AMT_ANNUITY                     0.036900
DAYS_LAST_PHONE_CHANGE          0.036306
AMT_GOODS_PRICE                 0.033145
AMT_INCOME_TOTAL                0.028730
REGION_POPULATION_RELATIVE      0.027273
HOUR_APPR_PROCESS_START         0.024056
OWN_CAR_AGE                     0.022309
YEARS_BEGINEXPLUATATION_MEDI    0.022224
LANDAREA_MEDI                   0.021471
BASEMENTAREA_MEDI               0.020348
YEARS_BUILD_MEDI                0.018238
COMMONAREA_MEDI                 0.018110
NONLIVINGAREA_MEDI              0.016516
OBS_30_CNT_SOCIAL_CIRCLE        0.016446
dtype: float64


In [18]:
feature_counts = [10, 20, 30, 40]
param_variations = [
    {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 10,   'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 15,   'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 20,   'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 30,   'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 15,   'min_samples_leaf': 5},
    {'n_estimators': 100, 'max_depth': 15,   'min_samples_leaf': 10},
    {'n_estimators': 200, 'max_depth': 15,   'min_samples_leaf': 5},
    {'n_estimators': 200, 'max_depth': 20,   'min_samples_leaf': 5},
    {'n_estimators': 200, 'max_depth': 20,   'min_samples_leaf': 10},
]

results = []

for n_features in feature_counts:
    top_features = importances.index[:n_features].tolist()
    X_subset = X_train[top_features]

    for params in param_variations:
        rf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
        scores = cross_val_score(rf, X_subset, y_train, cv=3, scoring='roc_auc')
        results.append({
            'n_features': n_features,
            **params,
            'roc_auc_mean': scores.mean(),
            'roc_auc_std':  scores.std(),
        })
        print(f"features={n_features}, params={params} → AUC={scores.mean():.4f} ± {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values('roc_auc_mean', ascending=False)
print("\nTop 10 combinations:")
print(results_df.head(10))


c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


features=10, params={'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1} → AUC=0.7089 ± 0.0022


c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


features=10, params={'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 1} → AUC=0.7348 ± 0.0026


c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


...

features=40, params={'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 5} → AUC=0.7428 ± 0.0016


c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


features=40, params={'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 5} → AUC=0.7397 ± 0.0021


c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\simon\Documents\ML\env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


features=40, params={'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 10} → AUC=0.7433 ± 0.0026

Top 10 combinations:
    n_features  n_estimators  max_depth  min_samples_leaf  roc_auc_mean  \
39          40           200       20.0                10      0.743337   
37          40           200       15.0                 5      0.742818   
36          40           100       15.0                10      0.742712   
26          30           100       15.0                10      0.742155   
27          30           200       15.0                 5      0.741991   
31          40           100       10.0                 1      0.741718   
29          30           200       20.0                10      0.741490   
21          30           100       10.0                 1      0.740784   
35          40           100       15.0                 5      0.740709   
25          30           100       15.0                 5      0.739987   

    roc_auc_std  
39     0.002631  
37     0.00